## Step 1: Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from delta.tables import *


Incremental Dataset Path

In [0]:
incremental_path="/Volumes/dbacademy/default/data/retail_delta_project/datasets/incremental"

##Step1:Read Silver Customer

In [0]:
silver_customers=spark.table("silver_customers")
display(silver_customers.limit(10))

customer_id,customer_name,city,segment,gender,signup_date,status,ingesttime,source_file,load_type
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00009,Customer_9,Delhi,Gold,F,2025-09-10,active,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch
C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive,2026-07-12T01:00:15.955Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/customers_batch.csv,batch


##Step2::Read CDC File

In [0]:
customer_cdc_path = f"{incremental_path}/day_2026-04-24/customers_cdc_2026-04-24.csv"

customer_cdc =spark.read\
    .format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load(customer_cdc_path)


###2.2:CDC File data exploration

In [0]:
display(customer_cdc.limit(10))

customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation
C00938,Customer_938,Mumbai,Platinum,Other,2025-08-02,active,2026-04-24,UPDATE
C01955,Customer_1955,Hyderabad,Silver,F,2025-03-14,inactive,2026-04-24,UPDATE
C01539,Customer_1539,Ahmedabad,Silver,M,2024-09-08,active,2026-04-24,UPDATE
C00577,Customer_577,Delhi,Gold,F,2025-11-17,inactive,2026-04-24,UPDATE
C01999,Customer_1999,Ahmedabad,Silver,M,2024-11-29,active,2026-04-24,UPDATE
C00948,Customer_948,Mumbai,Silver,M,2024-06-02,active,2026-04-24,UPDATE
C02069,Customer_2069,Chennai,Silver,Other,2025-03-15,active,2026-04-24,UPDATE
C01445,Customer_1445,Jaipur,Silver,M,2025-07-23,inactive,2026-04-24,UPDATE
C01135,Customer_1135,Jaipur,Regular,F,2024-06-22,active,2026-04-24,UPDATE
C01503,Customer_1503,Kolkata,Platinum,Other,2024-09-24,active,2026-04-24,UPDATE


In [0]:
customer_cdc.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- operation: string (nullable = true)



In [0]:
print("CDC Count:", customer_cdc.count())

CDC Count : 210


##Step3:Create Initial Dimension Table 

###3.1:Create Columns

In [0]:
window=Window.orderBy("customer_id")

dim_customer=(
    silver_customers\
        .withColumn("customer_sk",row_number().over(window))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dim_customer.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- ingesttime: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_type: string (nullable = true)
 |-- customer_sk: integer (nullable = false)



In [0]:
dim_customer=dim_customer\
    .withColumn("effective_date",current_date())\
        .withColumn("end_date",lit(None).cast("date"))\
            .withColumn("is_current",lit(True))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###3.2:Select Columns 

In [0]:
dim_customer = dim_customer.select(
    "customer_sk",
    "customer_id",
    "customer_name",
    "city",
    "segment",
    "gender",
    "signup_date",
    "status",
    "effective_date",
    "end_date",
    "is_current"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


##Step4:Save Customer Dimension Table

In [0]:
dim_customer.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("dim_customers_scd2")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(spark.table("dim_customers_scd2").limit(10))

customer_sk,customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,end_date,is_current
1,C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,2026-07-12,null,true
2,C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,2026-07-12,null,true
3,C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,2026-07-12,null,true
4,C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,2026-07-12,null,true
5,C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,2026-07-12,null,true
6,C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active,2026-07-12,null,true
7,C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active,2026-07-12,null,true
8,C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive,2026-07-12,null,true
9,C00009,Customer_9,Delhi,Gold,F,2025-09-10,active,2026-07-12,null,true
10,C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive,2026-07-12,null,true


## Step5:SCD Type 2

###5.1:Record to be Insert

In [0]:
customer_insert=customer_cdc.filter(
    col("operation")=="INSERT"
)

###5.2:Records to be update

In [0]:
customer_update=customer_cdc.filter(
    col("operation")=="UPDATE"
)

###5.3:Remove Duplicates

In [0]:
window = Window.partitionBy("customer_id").orderBy(desc("effective_date"))

customer_update = (
    customer_cdc
    .filter(col("operation")=="UPDATE")
    .withColumn("rn", row_number().over(window))
    .filter("rn = 1")
    .drop("rn")
)

In [0]:
customer_insert = (
    customer_cdc
    .filter(col("operation")=="INSERT")
    .withColumn("rn", row_number().over(window))
    .filter("rn = 1")
    .drop("rn")
)

In [0]:
print("Insert:",customer_insert.count())
print("Update:",customer_update.count())

Insert: 80
Update: 120


In [0]:
window = Window.partitionBy("customer_id").orderBy(desc("effective_date"))

customer_update = (
    customer_update
    .withColumn("rn", row_number().over(window))
    .filter("rn = 1")
    .drop("rn")
)

###5.4:Close Update Records

In [0]:
deltaTable = DeltaTable.forName(spark,"dim_customers_scd2")

(deltaTable.alias("target")
    .merge(
        customer_update.alias("source"),
        "target.customer_id = source.customer_id AND target.is_current = true"
    ).whenMatchedUpdate(
        set={
            "is_current": lit(False),
            "end_date": expr("date_sub(source.effective_date,1)")
        }
    ).execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print(
    spark.table("dim_customers_scd2")
    .filter("is_current=false")
    .count()
)

119


In [0]:
print(spark.table("dim_customers_scd2").count())

2475


###5.5:Insert new update records

In [0]:
max_sk = spark.table("dim_customers_scd2") \
    .agg(max("customer_sk")) \
    .collect()[0][0]

In [0]:
window = Window.orderBy("customer_id")

updated_records = (
    customer_update
    .dropDuplicates(["customer_id"])
    .withColumn("customer_sk", row_number().over(window) + max_sk)
    .withColumn("end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
updated_records.select(
    "customer_sk",
    "customer_id",
    "customer_name",
    "city",
    "segment",
    "gender",
    expr("try_cast(signup_date as date)").alias("signup_date"),
    "status",
    "effective_date",
    "end_date",
    "is_current"
).write \
.format("delta") \
.mode("append") \
.saveAsTable("dim_customers_scd2")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print("Total:", spark.table("dim_customers_scd2").count())

print("Historical:", spark.table("dim_customers_scd2")\
      .filter("is_current=false")\
      .count())

print("Current:",spark.table("dim_customers_scd2")\
      .filter("is_current=true")\
      .count())

Total: 2595
Historical: 119
Current: 2476


###5.6:Insert new  records

In [0]:

max_sk =  spark.table("dim_customers_scd2")\
    .agg(max("customer_sk"))\
    .collect()[0][0]


In [0]:
existing_customer = spark.table("dim_customers_scd2")\
    .select("customer_id")\
    .distinct()


In [0]:
new_customers = customer_insert\
    .join(existing_customer,
          "customer_id",
          "left_anti")


In [0]:
window = Window.orderBy("customer_id")

new_customers = new_customers\
    .withColumn(
        "customer_sk",
        row_number().over(window) + max_sk
    )\
    .withColumn("end_date", lit(None).cast("date"))\
    .withColumn("is_current", lit(True))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


##Step6:Save Intial Customers Dimension Table

In [0]:
new_customers.select(
    "customer_sk",
    "customer_id",
    "customer_name",
    "city",
    "segment",
    "gender",
    "signup_date",
    "status",
    "effective_date",
    "end_date",
    "is_current"
).write \
.format("delta") \
.mode("append") \
.saveAsTable("dim_customers_scd2")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print("Total Records :",spark.table("dim_customers_scd2").count())

print("Current Records :", spark.table("dim_customers_scd2")\
      .filter("is_current = true")\
      .count())

print("Historical Records :", spark.table("dim_customers_scd2")\
      .filter("is_current = false")\
      .count())

Total Records : 2675
Current Records : 2556
Historical Records : 119


#Function for incremental load(SCD Type 2)

In [0]:
def process_customer_cdc(cdc_path):

   
    customer_cdc =  spark.read\
        .option("header", True)\
        .option("inferSchema", True)\
        .csv(cdc_path)
    

    window = Window.partitionBy("customer_id", "operation") \
        .orderBy(desc("effective_date"))


    customer_update =  customer_cdc\
        .filter(col("operation") == "UPDATE")\
        .withColumn("rn", row_number().over(window))\
        .filter("rn = 1")\
        .drop("rn")
    

    
    customer_insert = customer_cdc\
        .filter(col("operation") == "INSERT")\
        .withColumn("rn", row_number().over(window))\
        .filter("rn = 1")\
        .drop("rn")
    

    from delta.tables import DeltaTable

    deltaTable = DeltaTable.forName(spark, "dim_customers_scd2")

    
    deltaTable.alias("target")\
            .merge(
            customer_update.alias("source"),
            "target.customer_id = source.customer_id AND target.is_current = true"
        )\
        .whenMatchedUpdate(
            set={
                "is_current": lit(False),
                "end_date": expr("date_sub(try_cast(source.effective_date as date),1)")
            }
        )\
        .execute()
    

    max_sk = spark.table("dim_customers_scd2")\
        .agg(max("customer_sk"))\
        .collect()[0][0]
    

    window = Window.orderBy("customer_id")

    updated_customers =customer_update\
        .join(
            spark.table("dim_customers_scd2").select("customer_id"),
            "customer_id",
            "inner"
        )\
        .dropDuplicates(["customer_id"])\
        .withColumn("customer_sk", row_number().over(window) + max_sk)\
        .withColumn("ingesttime", current_timestamp())\
        .withColumn("source_file", lit("customers_cdc_2026-04-24.csv"))\
        .withColumn("load_type", lit("INCREMENTAL"))\
        .withColumn("end_date", lit(None).cast("date"))\
        .withColumn("is_current", lit(True))
    

    updated_customers.select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        expr("try_cast(signup_date as date)").alias("signup_date"),
        "status",
        expr("try_cast(effective_date as date)").alias("effective_date"),
        "end_date",
        "is_current"
    ).write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("dim_customers_scd2")

    max_sk = spark.table("dim_customers_scd2")\
        .agg(max("customer_sk"))\
        .collect()[0][0]
    

    existing_customer = spark.table("dim_customers_scd2")\
        .select("customer_id")\
        .distinct()
    

    new_customers = customer_insert\
        .join(
            existing_customer,
            "customer_id",
            "left_anti"
        )\
        .withColumn(
            "customer_sk",
            row_number().over(window) + max_sk
        )\
        .withColumn("end_date", lit(None).cast("date"))\
        .withColumn("is_current", lit(True))
    

    new_customers.select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        expr("try_cast(signup_date as date)").alias("signup_date"),
        "status",
        expr("try_cast(effective_date as date)").alias("effective_date"),
        "end_date",
        "is_current"
    ).write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("dim_customers_scd2")

    print("Completed :", cdc_path)

# Day-26 Incremental Records

In [0]:
process_customer_cdc("/Volumes/dbacademy/default/data/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-26.csv")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Completed : /Volumes/dbacademy/default/data/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv


In [0]:
print("Total Records :",spark.table("dim_customers_scd2").count())\

print("Current Records :", spark.table("dim_customers_scd2")\
      .filter("is_current = true")
      .count())

print("Historical Records :",spark.table("dim_customers_scd2")\
      .filter("is_current = false")
      .count())

Total Records : 2873
Current Records : 2636
Historical Records : 237
